# 07 — DOE — fractional 2^(7−p)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ai-systems-today/cubespec/blob/main/notebooks/07_doe_fractional.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/ai-systems-today/cubespec/main?urlpath=lab/tree/notebooks/07_doe_fractional.ipynb) [![nbviewer](https://img.shields.io/badge/render-nbviewer-orange)](https://nbviewer.org/github/ai-systems-today/cubespec/blob/main/notebooks/07_doe_fractional.ipynb) [![Dashboard](https://img.shields.io/badge/open-dashboard-2563eb)](https://sensitive-spark.lovable.app)

**Abstract.** Compare 1/2, 1/4 and 1/8 fractional factorials against the full 2⁷ and decide when the run-count saving is safe.

**Mirrors.** **DOE** tab → design selector + generators/resolution panel.


In [ ]:
# cubespec-bootstrap
# Install cubespec on first run (Colab/Binder safe; no-op if already importable).
try:
    import cubespec  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'git+https://github.com/ai-systems-today/cubespec.git'])
    import cubespec  # noqa: F401


## 1. Trading runs for resolution

A 2^(7−p) fractional factorial uses `2^(7-p)` runs instead of `128`. The price is **aliasing**: some main effects become indistinguishable from interactions. We compare the 1/2, 1/4 and 1/8 fractions against the full design and ask: do we still pick the same top drivers?


In [ ]:
from cubespec import DEFAULT_CSP, full_factorial, fractional_factorial, main_effects
import pandas as pd

designs = {
    'full 2⁷':         full_factorial(DEFAULT_CSP, levels=2),
    'fractional 1/2':  fractional_factorial(DEFAULT_CSP, fraction='1/2'),
    'fractional 1/4':  fractional_factorial(DEFAULT_CSP, fraction='1/4'),
    'fractional 1/8':  fractional_factorial(DEFAULT_CSP, fraction='1/8'),
}
summary = pd.DataFrame({name: [len(df)] for name, df in designs.items()},
                       index=['runs'])
summary


## 2. Side-by-side ranking of P9 main effects


In [ ]:
rows = []
for name, df in designs.items():
    e = (main_effects(df)
         .query("output == 'P9_compressive_strength'")
         .set_index('factor')['effect'].rename(name))
    rows.append(e)
merged = pd.concat(rows, axis=1)
merged.sort_values('full 2⁷', key=abs, ascending=False)


In [ ]:
import matplotlib.pyplot as plt
ax = merged.sort_values('full 2⁷', key=abs, ascending=True).plot.barh(
    figsize=(8, 4.2), color=['#2952b3', '#5b8def', '#9bb6f0', '#e94f64'])
ax.set_xlabel('main effect on P9 (MPa)')
ax.set_title('Full vs fractional designs — main-effect agreement')
plt.tight_layout(); plt.show()


## 3. Decision rule

If your top-3 drivers under 1/8 fractional agree with the full design, the 16-run design is sufficient for screening. If they disagree, fall back to 1/4 (32 runs) or full.


In [ ]:
# cubespec-reproducibility-footer
import platform, sys, numpy as np, scipy, pandas as pd, cubespec
print('Reproducibility')
print('  cubespec :', cubespec.__version__)
print('  python   :', sys.version.split()[0], 'on', platform.system())
print('  numpy    :', np.__version__)
print('  scipy    :', scipy.__version__)
print('  pandas   :', pd.__version__)
